# S8 · Spark Structured Streaming y ventanas de tiempo

**Integrante** · Joel Jonatan Quispe Mamani

**Dimensión** · Velocidad mediana en vuelo

Equipo 04 · AeroStreamPeru · Ciclo 9

## 1. Lectura, deduplicación y ventana

El [proceso de Spark](procesar_ventanas.py) lee la telemetría y las confirmaciones desde Kafka con `readStream`. La primera consulta aplica `dropDuplicatesWithinWatermark`. La segunda reúne los eventos en ventanas fijas UTC de diez minutos y publica las ventanas cerradas. Cada consulta guarda su propio checkpoint. El margen es de cuatro minutos, equivalente a dos intervalos de captura.

In [1]:
from pathlib import Path
import json
import sys

SESION = Path("/opt/s08-streaming-estructurado")
sys.path.insert(0, str(SESION))
from procesar_ventanas import ESQUEMA_EVENTO, WATERMARK
from probar_s08 import leer_registros

MODO = "prueba"  # Usa "vivo" para consultar la salida del proceso continuo.
if MODO == "prueba":
    ruta = SESION / "artifacts" / "evidencia_s08.json"
    if not ruta.is_file():
        raise FileNotFoundError("Ejecuta probar_s08.py con el comando del README de S8.")
    evidencia = json.loads(ruta.read_text(encoding="utf-8"))
    topico = evidencia["topico_ventanas"]
else:
    topico = "opensky.ventanas.v1"

print(f"Tópico de salida: {topico}")
print(f"Esquema de entrada: {ESQUEMA_EVENTO.simpleString()}")
print(f"Watermark configurado: {WATERMARK}")

Tópico de salida: opensky.s08-prueba-20261124172640.ventanas.v1
Esquema de entrada: struct<time:bigint,icao24:string,lat:double,lon:double,onground:boolean,velocity:double,baroaltitude:double,vertrate:double,lastposupdate:double,lastcontact:double,version_esquema:int,capture_id:string,event_id:string,tipo:string,id_registro:string,estado:string,estados_publicados:int>
Watermark configurado: 4 minutes


## 2. Evidencia de la consulta

El modo de prueba consulta el tópico aislado que dejó `probar_s08.py`. La prueba reproduce estados reales del Parquet y controla el orden de llegada. También incluye confirmaciones de captura con cero, fallida e incompleta para verificar el significado de la cobertura.

In [2]:
registros = leer_registros(topico, maximo_por_particion=100)
ventanas = {ventana["window_id"]: ventana for ventana in registros}
if not ventanas:
    raise RuntimeError("Todavía no hay ventanas cerradas en este tópico.")
for ventana in sorted(ventanas.values(), key=lambda fila: fila["inicio_utc"])[-5:]:
    assert ventana["capturas_observadas"] + ventana["capturas_faltantes"] == 5
    print(
        ventana["inicio_utc"],
        f"capturas={ventana['capturas_observadas']}/5",
        f"velocidad_mediana_t={ventana['velocidad_mediana_t']}",
    )

if MODO == "prueba":
    assert evidencia["duplicados_eliminados"] >= 1
    assert evidencia["registros_fuera_del_watermark"] >= 1
    assert evidencia["checkpoint_recuperado"]
    assert evidencia["ventana_completa"] in ventanas
    print("Duplicados descartados:", evidencia["duplicados_eliminados"])
    print("Registros fuera del watermark:", evidencia["registros_fuera_del_watermark"])
    print("Checkpoint recuperado:", evidencia["checkpoint_recuperado"])

2026-01-04T00:00:00.000Z capturas=3/5 velocidad_mediana_t=235.1211696914452
2026-01-04T00:10:00.000Z capturas=1/5 velocidad_mediana_t=None
2026-01-04T00:20:00.000Z capturas=5/5 velocidad_mediana_t=234.7184223857197
Duplicados descartados: 1
Registros fuera del watermark: 12
Checkpoint recuperado: True


## 3. Interpretación de mi indicador

La mediana se calcula sobre velocidades válidas de estados en vuelo. Si la ventana solo contiene una captura exitosa sin aeronaves, la velocidad queda nula. Esa diferencia permite interpretar la disponibilidad de mediciones.

La salida Kafka se identifica por `window_id`. Si un fallo entre escritura y confirmación obliga a reintentar un micro-lote, el consumidor puede usar esa clave para reconocer salidas repetidas.